<a href="https://colab.research.google.com/github/ebuseyy/SGuA-projects/blob/main/BeforeRevision/ABC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# Define the number of relays and the number of employed bees
num_relays = 6
num_employed_bees = 100

# Define the maximum and minimum values for parameters
ti_min = 1.0
ti_max = 2.2
td_min = 0.2
td_max = 1.0
Ikd1 = 1263
Ikd2_6 = 5639
Ip_min = [117.152, 523, 400, 500, 600, 400]
Ip_max = [128.65, 575, 420, 510, 610, 420]

def calculate_ti(td, Ikd, RIP):
    ti = [0] * num_relays
    for k in range(num_relays):
        x = (Ikd[k] / RIP[k]) ** 0.02
        calculated_ti = 0.14 * td[k] / (x - 1)
        # Soft constraint handling
        if calculated_ti < ti_min:
            ti[k] = ti_min + (ti_min - calculated_ti) * 0.1  # Penalize but don't set to boundary
        elif calculated_ti > ti_max:
            ti[k] = ti_max - (calculated_ti - ti_max) * 0.1  # Penalize but don't set to boundary
        else:
            ti[k] = calculated_ti
    return ti

def evaluate_solution(td, Ikd, RIP):
    # Ensure td is within the specified range (Constraint 2)
    if any(t < td_min or t > td_max for t in td):
        return float('inf')  # Penalize invalid solutions

    # Ensure Ikd is within the specified range
    if any(Ikd_val < Ikd2_6 for Ikd_val in Ikd[1:]):
        return float('inf')  # Penalize invalid solutions

    # Ensure Ip is within the specified range (Constraint 3)
    for i in range(num_relays):
        if RIP[i] < Ip_min[i] or RIP[i] > Ip_max[i]:
            return float('inf')  # Penalize invalid solutions

    # Calculate relay operation times
    relay_times = calculate_ti(td, Ikd, RIP)

    # Check if relay_times are within the specified range (Constraint 1)
    for i in range(num_relays):
        if relay_times[i] < ti_min or relay_times[i] > ti_max:
            return float('inf')  # Penalize invalid solutions

    # Apply the second constraint
    # t1 should be greater than t2 by at least 0.3 seconds
    if relay_times[0] <= relay_times[1] + 0.3:
        return float('inf')

    # Apply the updated constraint: t2 should be greater than other relays (t3, t4, t5, t6) by at least 0.3 seconds
    min_other_relays = min(relay_times[2:])  # Find the minimum of t3, t4, t5, t6
    if relay_times[1] <= min_other_relays + 0.3:
        return float('inf')

    # Minimize the sum of relay operation times
    return sum(relay_times)

def run_abc_algorithm(num_iterations=1000, limit=10):
    # Initialize employed bees with random solutions
    employed_solutions = []
    for _ in range(num_employed_bees):
        td = np.linspace(td_min, td_max, num_relays) + np.random.uniform(-0.1, 0.1, size=num_relays)
        td = np.maximum(td_min, np.minimum(td, td_max))# Ensure that td values are within bounds
        Ikd = [Ikd1] + [Ikd2_6] * (num_relays - 1)
        RIP = [np.random.uniform(low=Ip_min[i], high=Ip_max[i]) for i in range(num_relays)]
        solution = {
            'td': td,
            'Ikd': Ikd,
            'RIP': RIP,
            'fitness': evaluate_solution(td, Ikd, RIP)
        }
        employed_solutions.append(solution)

    # Initialize trial counter for each solution
    trial_counter = np.zeros(num_employed_bees)

    # Main ABC loop
    for iteration in range(num_iterations):
        # Employed bees phase
        for i in range(num_employed_bees):
            # Select a random dimension to update
            dimension_to_update = np.random.randint(0, num_relays)

            # Generate a new solution by perturbing the current one
            new_td = employed_solutions[i]['td'].copy()
            new_td[dimension_to_update] = np.random.uniform(low=td_min, high=td_max)
            new_Ikd = employed_solutions[i]['Ikd'].copy()
            new_RIP = employed_solutions[i]['RIP'].copy()
            new_RIP[dimension_to_update] = np.random.uniform(low=Ip_min[dimension_to_update], high=Ip_max[dimension_to_update])
            new_fitness = evaluate_solution(new_td, new_Ikd, new_RIP)

            # Update the employed bee's solution if it's better
            if new_fitness < employed_solutions[i]['fitness']:
                employed_solutions[i]['td'] = new_td
                employed_solutions[i]['Ikd'] = new_Ikd
                employed_solutions[i]['RIP'] = new_RIP
                employed_solutions[i]['fitness'] = new_fitness
                trial_counter[i] = 0
            else:
                trial_counter[i] += 1

        # Onlooker bees phase
        # Calculate probabilities inversely to fitness
        fitness_values = np.array([1 / (1 + solution['fitness']) if solution['fitness'] >= 0 else 1 + abs(solution['fitness']) for solution in employed_solutions])

        # Handle invalid probabilities
        if np.isfinite(fitness_values.sum()) and fitness_values.sum() > 0:
            probabilities = fitness_values / fitness_values.sum()
        else:
            # Avoid division by zero or invalid values
            probabilities = np.ones(num_employed_bees) / num_employed_bees  # Equal probability for all

        for _ in range(num_employed_bees):
            # Select a solution based on probabilities
            selected_index = np.random.choice(num_employed_bees, p=probabilities)
            selected_solution = employed_solutions[selected_index]

            # Employ a modification strategy (similar to the employed bees phase)
            dimension_to_update = np.random.randint(0, num_relays)
            new_td = selected_solution['td'].copy()
            new_td[dimension_to_update] = np.random.uniform(low=td_min, high=td_max)
            new_Ikd = selected_solution['Ikd'].copy()
            new_RIP = selected_solution['RIP'].copy()
            new_RIP[dimension_to_update] = np.random.uniform(low=Ip_min[dimension_to_update], high=Ip_max[dimension_to_update])
            new_fitness = evaluate_solution(new_td, new_Ikd, new_RIP)

            # Update the onlooker bee's solution if it's better
            if new_fitness < selected_solution['fitness']:
                employed_solutions[selected_index]['td'] = new_td
                employed_solutions[selected_index]['Ikd'] = new_Ikd
                employed_solutions[selected_index]['RIP'] = new_RIP
                employed_solutions[selected_index]['fitness'] = new_fitness
                trial_counter[selected_index] = 0
            else:
                trial_counter[selected_index] += 1

        # Scout bees phase
        for i in range(num_employed_bees):
            if trial_counter[i] > limit:
                # Generate a new random solution
                td = np.random.uniform(low=td_min, high=td_max, size=num_relays)
                Ikd = [Ikd1] + [Ikd2_6] * (num_relays - 1)
                RIP = [np.random.uniform(low=Ip_min[j], high=Ip_max[j]) for j in range(num_relays)]
                employed_solutions[i] = {
                    'td': td,
                    'Ikd': Ikd,
                    'RIP': RIP,
                    'fitness': evaluate_solution(td, Ikd, RIP)
                }
                trial_counter[i] = 0

    # Find the best solution
    best_solution = min(employed_solutions, key=lambda x: x['fitness'])

    # Calculate relay operating times using the provided calculate_ti() function
    relay_ti = calculate_ti(best_solution['td'], Ikd, best_solution['RIP'])

    return best_solution, relay_ti

# Call the ABC algorithm function
best_solution, relay_ti = run_abc_algorithm()

# Print the best solution and relay operating times
print("Table 1. ABC Results.")
print("{:<10} {:<12} {:<10} {:<10}".format("Relay", "td", "Ip(A)", "Ti(sn)"))
for i in range(num_relays):
    print("{:<10} {:<12.6f} {:<10.3f} {:<10.3f}".format("Relay-{}".format(i + 1), best_solution['td'][i], best_solution['RIP'][i], relay_ti[i]))
print("Best Fitness (minimized relay operation times):", best_solution['fitness'])


Table 1. ABC Results.
Relay      td           Ip(A)      Ti(sn)    
Relay-1    0.607426     122.098    1.778     
Relay-2    0.500833     552.085    1.474     
Relay-3    0.380498     407.456    1.001     
Relay-4    0.201098     504.325    1.043     
Relay-5    0.252597     605.568    1.023     
Relay-6    0.376561     407.736    1.002     
Best Fitness (minimized relay operation times): 7.320720107688195
